# Chapter 6 Lab — Semantics and Discourse

A compact, public-domain English corpus (a handful of short paragraphs, not the original
lecture material's 2GB Russian news dump) for a distributional-semantics recap, followed by
rule-based coreference resolution.

## 1. Small distributional-semantics recap

In [ ]:
paragraphs = [
    "The king ruled the kingdom with wisdom and fairness.",
    "The queen ruled the kingdom after the king passed away.",
    "A man walked into the store to buy bread.",
    "A woman walked into the store to buy milk.",
    "The prince inherited the throne from his father the king.",
]
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer

vec = CountVectorizer(token_pattern=r"[a-zA-Z']+")
X = vec.fit_transform(paragraphs).toarray()
vocab = vec.get_feature_names_out()
print(vocab)

## 2. Rule-based coreference resolution with spaCy

In [ ]:
import spacy
try:
    nlp = spacy.load("en_core_web_sm")
    spacy_mode = "en_core_web_sm"
except Exception as exc:
    nlp = spacy.blank("en")
    nlp.add_pipe("sentencizer")
    spacy_mode = f"blank English fallback ({type(exc).__name__})"
print("spaCy mode:", spacy_mode)

def naive_coref(doc):
    """Toy resolver: use POS/deps when available, otherwise simple nearest-noun heuristics."""
    mentions, pronouns = [], []
    fallback_nouns = {"maria", "store", "book", "trophy", "suitcase"}
    for tok in doc:
        text = tok.text.lower().strip(".,")
        is_mention = tok.pos_ in ("PROPN", "NOUN") and tok.dep_ in ("nsubj", "dobj", "pobj")
        if not tok.pos_ and text in fallback_nouns:
            is_mention = True
        if is_mention:
            mentions.append(tok)
        if text in ("she", "he", "it", "they"):
            candidates = [m for m in mentions if m.i < tok.i]
            pronouns.append((tok, candidates[-1] if candidates else None))
    return pronouns

text = "Maria went to the store. She bought a book. It was expensive."
doc = nlp(text)
for pron, antecedent in naive_coref(doc):
    print(f"'{pron.text}' -> '{antecedent.text if antecedent else '?'}'")


## 3. A Winograd-Schema style stress test

In [ ]:
s1 = "The trophy didn't fit in the suitcase because it was too big."
s2 = "The trophy didn't fit in the suitcase because it was too small."
for s in (s1, s2):
    d = nlp(s)
    print(s, "->", [(pron.text, ant.text if ant else "?") for pron, ant in naive_coref(d)])


## Exercise

The `naive_coref` resolver always picks the nearest preceding noun — it will get both Winograd
sentences above wrong in the same way (it has no notion of "big" implying "didn't fit"). What
kind of model or feature would you need to add to fix this?